# 02.3 — Assembling the prompt and generating an answer

Retrieval gives you chunks. The user wants a sentence. This notebook closes
that gap, and it's shorter than you'd expect — the 'generation' half of
retrieval-augmented generation is mostly string formatting.

The important cell is the one that prints the prompt. Most people building RAG
never look at what they're actually sending, and almost every strange answer is
explained by looking.

In [1]:
# Install if using jupyterlab locally on cpu
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

In [2]:
!pip install -q pymupdf4llm sentence-transformers openai

## Rebuild the retriever

Notebooks 1 and 2 in one cell. Nothing new here — skip past it.

In [3]:
from pathlib import Path
import numpy as np
import pymupdf4llm
from sentence_transformers import SentenceTransformer

CORPUS = Path('../../corpus/docs')
NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]


def chunk(text, size=500):
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks = []
for name in NATIVE_PDFS:
    text = pymupdf4llm.to_markdown(str(CORPUS / name))
    for i, body in enumerate(chunk(text)):
        chunks.append({'doc': name, 'n': i, 'text': body})

model = SentenceTransformer('BAAI/bge-small-en-v1.5')
vectors = model.encode([c['text'] for c in chunks], normalize_embeddings=True, show_progress_bar=True)


def search(query, k=3):
    qv = model.encode([query], normalize_embeddings=True)[0]
    scores = vectors @ qv
    return [chunks[i] for i in np.argsort(-scores)[:k]]


print(f'{len(chunks)} chunks ready')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

71 chunks ready


## The prompt is just a string

There is no magic step between retrieval and the model. You paste the chunks
into some text, add the question, and send it.

Three things are doing work in the template below:

- the retrieved chunks, labelled with where they came from
- an instruction to answer *only* from those chunks
- permission to say it doesn't know

That last one matters more than it looks. Without it a model will answer
anyway, because answering is what it was trained to do.

In [4]:
PROMPT = '''Answer the question using only the context below.
If the context does not contain the answer, say you don't know.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''


def build_prompt(question, k=3):
    retrieved = search(question, k)
    context = '\n\n'.join(
        f'[{c["doc"]} #{c["n"]}]\n{c["text"]}' for c in retrieved
    )
    return PROMPT.format(context=context, question=question)

### Look at one

Run this and read the whole thing. It's the single most useful habit in this
course.

In [5]:
print(build_prompt('What is the maximum emergency procurement without competitive sourcing?'))

Answer the question using only the context below.
If the context does not contain the answer, say you don't know.

CONTEXT:
[sahel-procurement-policy-v3.pdf #4]
5,000,000 without competitive sourcing. Emergency procurement must be reported to the Management Procurement Committee at its next meeting with a written justification. 

Emergency procurement may not be used for recurring requirements that could reasonably have been anticipated. 

## **5. Conflict of interest** 

Staff participating in a sourcing decision shall declare any relationship with a bidder, including relationships by marriage, prior employment within the last three years, and any fina

[sahel-procurement-policy-v3.pdf #1]
parately by the Group Technology Procurement Standard. 

## **2. Approval thresholds** 

|**Value (NGN)**|**Sourcing requirement**|**Approval authority**|
|---|---|---|
|Up to 500,000|Single quotation|Branch Manager or Unit Head|
|500,001 – 2,500,000|Three written quotations|Head, Administration|
|2

Some things worth noticing in that output.

**This is everything the model gets.** It has no access to the corpus, no
memory of earlier notebooks, no ability to go and check. If the answer isn't in
that text, it isn't available.

**Two of the three chunks are probably irrelevant.** We asked for the top 3 and
got 3, regardless of whether 3 were any good. Retrieval always returns `k`
results — there's no threshold saying 'nothing here was close enough'.

**The chunks are ragged.** Mid-sentence starts and ends, from the fixed-size
splitting in notebook 1. The model has to work around that.

**Order is arbitrary from the model's point of view** but not harmless. Models
attend more reliably to the beginning and end of a long context than the
middle. Ours is short enough not to matter yet. Module 10 is about when it
does.

## Send it

We're using OpenRouter, which speaks the OpenAI API. One key covers generation,
and swapping models is one string.

In [6]:
import os
from openai import OpenAI

# In Docker this comes from .env. On Colab, use the secrets panel:
#     from google.colab import userdata
#     os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

assert os.environ.get('OPENROUTER_API_KEY'), 'set OPENROUTER_API_KEY first'

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url='https://openrouter.ai/api/v1',
)

CHAT_MODEL = 'poolside/laguna-xs-2.1:free' # Check https://openrouter.ai/models for free models

In [7]:
def answer(question, k=3, show_prompt=False):
    prompt = build_prompt(question, k)
    if show_prompt:
        print(prompt)
        print('=' * 70)

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    return response.choices[0].message.content

`temperature=0` asks for the most predictable output the model can give. It
does not make it deterministic — run the same question twice and you may get
different words. That inconvenient fact is why module 12 spends time on how to
evaluate something that won't sit still.

### A question it should get right

In [8]:
print(answer('What is the maximum emergency procurement without competitive sourcing?'))

The maximum emergency procurement without competitive sourcing is **5,000,000** (as stated in the context: "5,000,000 without competitive sourcing"). Emergency procurement must still be reported to the Management Procurement Committee with a written justification.


That's a complete RAG system. Roughly forty lines.

It also has every problem from notebook 2 still in it — they're just harder to
see now, because a fluent paragraph looks like an answer whether or not it is
one.

## Now watch it be confidently wrong

Notebook 2 showed that both editions of the handbook are in the index. Here's
what happens when the model reads them together.

In [9]:
print(answer('How many days of annual leave do confirmed staff get?', k=4, show_prompt=True))

Answer the question using only the context below.
If the context does not contain the answer, say you don't know.

CONTEXT:
[sahel-employee-handbook-2023.pdf #5]
ously. 

Staff at Assistant Manager grade and above are entitled to an additional three working days. 

## **5. Other leave** 

**Maternity leave** of sixteen weeks is granted at full pay to female staff who have completed twelve months of continuous service. **Paternity leave** of five working days is granted within two months of the birth. **Compassionate leave** of up to five working days is granted on the death of a spouse, child, parent or sibling. **Study leave** may be granted for profess

[sahel-employee-handbook-2025.pdf #5]
eously. 

Staff at Assistant Manager grade and above are entitled to an additional five working days. 

## **5. Other leave** 

**Maternity leave** of sixteen weeks is granted at full pay to female staff who have completed twelve months of continuous service. **Paternity leave** of ten working day

Read the prompt above the answer, then the answer.

You'll get one of three outcomes, and all three are bad in different ways:

- **It picks the 2023 number.** Confidently wrong, and it can cite a real
  document to support it.
- **It picks 2025.** Right, by luck, not because anything told it which edition
  was current.
- **It reports both and hedges.** Most honest, least useful — the user has to
  work out which applies.

Nothing here is a hallucination. The model read what it was given and reported
it faithfully. The bug happened long before generation: nothing in the pipeline
records that one document replaces the other.

This is why 'the LLM got it wrong' is almost never the right diagnosis. Module
11 covers separating these cases properly.

## And watch it handle a question it can't answer

In [10]:
print(answer('What was revenue in 2020?'))

The context does not provide the specific revenue figure for 2020. While it mentions a "six-year performance" chart covering 2019–2024, the actual data points for each year (including 2020) are not included in the provided text. The table referenced in the context only lists data for 2024 and does not include historical revenue figures for earlier years. Therefore, the answer cannot be determined from the given context. 

ANSWER: I don't know.


The figure exists only inside a chart image in the annual report, so no chunk
contains it. The prompt gives the model permission to say it doesn't know, so
with luck it does.

Try removing that line from `PROMPT` and running again. Models will usually
produce a number when asked for one, and a plausible fabricated figure in a
financial answer is considerably worse than a refusal.

Getting a system to abstain reliably is harder than one instruction makes it
look. Module 11.

## What's next

You have a working RAG system and you've seen it fail three ways.

Everything so far has been judged by eye. That doesn't scale past a handful of
questions, and it quietly rewards whichever answer happens to read well.

Notebook 4 shows what a framework would have hidden from you here. Notebook 5
replaces your judgement with a number.